> **VERSIÓN ESTUDIANTES** — completen las celdas marcadas `# TU CÓDIGO AQUÍ`.

# Taller 2 — Mercados e interacciones estratégicas en redes

**MODELOS DE INTERACCIONES SOCIALES — ECON 64597, Sección 1**
Universidad de los Andes · Facultad de Economía · 2026-2
Profesor: Alvaro J. Riascos Villegas · Complementaria: Gabriela Zamora

**Entrega:** semana 35 (Clase 4) · **Peso:** 10% de la nota final
**Grupos:** mínimo 2, máximo 3 personas

**Referencia única: [EK] Easley & Kleinberg (2010), capítulos 10, 11 y 12.**
Texto completo: https://www.cs.cornell.edu/home/kleinber/networks-book/

---

## Objetivo

Los tres capítulos responden a la misma pregunta desde ángulos distintos: **¿cómo
determina la estructura de la red quién comercia con quién, a qué precio, y quién se
queda con el excedente?**

| Cap. | Situación | De dónde sale el poder |
|---|---|---|
| 10 | Mercado bipartito, sin intermediarios | De ningún lado: los precios de equilibrio son eficientes |
| 11 | Los intermediarios fijan los precios | De tener una **arista esencial** |
| 12 | Sin roles fijos, negociación bilateral | De las **opciones externas** que da la red |

Diez ejercicios en total, tres o cuatro por capítulo. Las celdas de teoría son
autocontenidas y siguen la notación del libro, pero **no reemplazan la lectura**.

**Convención de notación (la del libro).** Los **vendedores** se dibujan a la
izquierda y los **compradores** a la derecha. La valoración que el comprador $j$ tiene
por la casa del vendedor $i$ se escribe $v(i,j)$, y el precio del vendedor $i$ es
$p(i)$. Los conjuntos constreñidos son conjuntos de **compradores**.

## Instrucciones de entrega

1. Completen todas las celdas `# TU CÓDIGO AQUÍ` y todas las preguntas
   `TU RESPUESTA AQUÍ`. No borren las celdas de verificación (`assert`).
2. `Kernel → Restart & Run All` en kernel limpio antes de entregar.
   **Un notebook que no corre de principio a fin pierde 20 puntos.**
3. Entreguen un `.zip` autocontenido: notebook ejecutado, `requirements.txt` y un
   `README.md` de media página.
4. Aplica el protocolo de la sección 5.b del programa: auditoría con *AI code reviewer*
   y **micro-reunión** de defensa. *"No sé, lo escribió la IA"* cuenta como entrega
   incompleta.

## Rúbrica (100 puntos)

| Parte | Tema | Ejercicios | Puntos |
|---|---|---|---|
| 1 | Mercados de emparejamiento | 1.1 – 1.4 | 36 |
| 2 | Redes de comercio con intermediarios | 2.1 – 2.3 | 28 |
| 3 | Negociación y poder | 3.1 – 3.3 | 26 |
| 4 | Discusión escrita y Declaración de Uso de IA | — | 10 |


---
## Datos del grupo y Declaración de Uso de IA

El uso de herramientas de IA está **permitido**. Lo que se evalúa es su criterio.
Una declaración vacía o genérica descuenta los 4 puntos correspondientes.


In [23]:
GRUPO = {
    "integrantes": ["Nombre Apellido — 202512345", "Nombre Apellido — 202567890"],
    "seccion": 1,
    "fecha_entrega": "2026-XX-XX",
}

DECLARACION_IA = """
Herramientas usadas: (p. ej. Claude Opus 5, ChatGPT, GitHub Copilot; o "ninguna")

Dónde se usaron y para qué:
- Ejercicio X.Y: ...

Qué se verificó, corrigió o rechazó de la salida de la IA y por qué:
- ...

Declaramos que podemos explicar, modificar y defender todo el código entregado.
"""
for i in GRUPO["integrantes"]:
    print(i)

Nombre Apellido — 202512345
Nombre Apellido — 202567890


---
## Parte 0 — Preparación

Todos los datos son ejemplos del libro. No hay descargas ni archivos externos.

Se usa `Fraction` en la Parte 3 porque las condiciones de balance son igualdades
exactas ($1/3$, $2/3$, $1/4$, …) y con `float` fallarían por redondeo.


In [24]:
import itertools
from fractions import Fraction

import networkx as nx

print("networkx", nx.__version__)

def ok(mensaje):
    print("OK -", mensaje)

networkx 3.6.1


---
# Parte 1 — Mercados de emparejamiento ([EK] cap. 10) · 36 puntos

## 1.0 Teoría

### El escenario y los tres objetos

Hay $n$ **vendedores**, cada uno con una casa que valora en 0, y $n$ **compradores**,
cada uno de los cuales quiere exactamente una. Conviene distinguir con cuidado tres
objetos que el libro maneja y que es fácil confundir:

| Objeto | Qué es | ¿Interviene el grafo? |
|---|---|---|
| **Asignación** | Una biyección compradores ↔ vendedores. Hay $n!$ | No |
| **Emparejamiento perfecto** | Una asignación **compatible con las aristas** | Sí: restricción de factibilidad |
| **Asignación óptima** | Una asignación que **maximiza la valoración total** | No: criterio de optimalidad |

> **Emparejamiento perfecto ([EK] §10.1).** Una asignación de los nodos de un lado a
> los del otro tal que (i) cada nodo está conectado por una arista al nodo al que se
> le asigna, y (ii) no hay dos nodos asignados al mismo. Equivalentemente: una
> elección de aristas en la que cada nodo es extremo de **exactamente una**.

### Cuándo falla: conjuntos constreñidos

> **Vecindario.** Para un conjunto $S$ de compradores, $N(S)$ es el conjunto de
> vendedores con alguna arista hacia algún nodo de $S$.
>
> **Conjunto constreñido.** Un conjunto $S$ de compradores con $|N(S)| < |S|$.

> **Teorema del emparejamiento (Hall).** Existe un emparejamiento perfecto **si y solo
> si** no existe ningún conjunto constreñido.

El libro enuncia solo la dirección difícil (sin emparejamiento ⟹ hay constreñido). La
otra es inmediata por el principio del palomar: si $M$ es perfecto, su restricción a
$S$ es una inyección de $S$ en $N(S)$, luego $|S| \leq |N(S)|$.

### Valoraciones, precios y equilibrio

$v(i,j)$ es la valoración del comprador $j$ por la casa del vendedor $i$; $p(i)$ es el
precio del vendedor $i$. El **pago** del comprador $j$ si compra la casa de $i$ es

$$v(i,j) - p(i).$$

El **grafo de vendedores preferidos** conecta a cada comprador con los vendedores que
maximizan su pago, siempre que ese máximo sea $\geq 0$.

> **Precios de equilibrio** (*market-clearing*). Precios cuyo grafo de vendedores
> preferidos admite un emparejamiento perfecto.

Dos teoremas ([EK] §10.3–10.4): **existen** para cualquier conjunto de valoraciones, y
**producen la asignación óptima**. La demostración de lo segundo es corta y conviene
tenerla presente para el ejercicio 1.4. Para toda asignación,

$$\underbrace{\sum_j v(M(j), j)}_{\text{valoración total}}
 = \underbrace{\sum_j \big(v(M(j),j) - p(M(j))\big)}_{\text{pago de los compradores}}
 + \underbrace{\sum_i p(i)}_{\text{constante}}.$$

El último sumando **no depende de la asignación**, porque en toda asignación venden
todos los vendedores. Y en un emparejamiento del grafo de preferidos cada comprador
obtiene su pago máximo. Sumando un máximo y una constante: la valoración total es
máxima.


In [25]:
# Mercado de la figura 10.5 de [EK]: vendedores a, b, c; compradores x, y, z.
VENDEDORES = ["a", "b", "c"]
COMPRADORES = ["x", "y", "z"]

# Notación del libro: V[(i, j)] = v(i, j) = valoración del COMPRADOR j por la casa de i
V = {("a", "x"): 12, ("b", "x"): 4, ("c", "x"): 2,
     ("a", "y"):  8, ("b", "y"): 7, ("c", "y"): 6,
     ("a", "z"):  7, ("b", "z"): 5, ("c", "z"): 2}

for j in COMPRADORES:
    print(f"  v(·, {j}) =", {i: V[(i, j)] for i in VENDEDORES})

  v(·, x) = {'a': 12, 'b': 4, 'c': 2}
  v(·, y) = {'a': 8, 'b': 7, 'c': 6}
  v(·, z) = {'a': 7, 'b': 5, 'c': 2}


### Ejercicio 1.1 — Grafo de vendedores preferidos (8 puntos)

Implementen `grafo_preferidos(V, p, vendedores, compradores)`, que devuelva el grafo
bipartito correspondiente a la definición de arriba.

- Debe contener **todos** los nodos, aun los aislados. Marquen vendedores con
  `bipartite=0` y compradores con `bipartite=1`.
- Un comprador puede tener **varias** aristas si hay empate en el pago máximo.
- Si su pago máximo es **negativo**, no se le conecta ninguna arista.


In [26]:
def grafo_preferidos(V, p, vendedores, compradores):
    """Grafo bipartito comprador-vendedor(es) que maximizan el pago v(i,j) - p(i)."""
    G = nx.Graph()
    G.add_nodes_from(vendedores, bipartite=0)
    G.add_nodes_from(compradores, bipartite=1)
    for j in compradores:
        pagos = {i: V[(i, j)] - p[i] for i in vendedores}
        mejor_pago = max(pagos.values())
        if mejor_pago >= 0:
            for i in vendedores:
                if pagos[i] == mejor_pago:
                    G.add_edge(i, j)
    return G

In [27]:
# Con todos los precios en cero, los tres compradores prefieren la casa de a
p0 = {"a": 0, "b": 0, "c": 0}
G0 = grafo_preferidos(V, p0, VENDEDORES, COMPRADORES)
assert G0.number_of_nodes() == 6
assert sorted(G0["x"]) == ["a"] and sorted(G0["y"]) == ["a"] and sorted(G0["z"]) == ["a"]

# Con los precios (3, 1, 0) aparecen empates
p1 = {"a": 3, "b": 1, "c": 0}
G1 = grafo_preferidos(V, p1, VENDEDORES, COMPRADORES)
assert sorted(G1["x"]) == ["a"]           # pagos 9, 3, 2
assert sorted(G1["y"]) == ["b", "c"]      # pagos 5, 6, 6
assert sorted(G1["z"]) == ["a", "b"]      # pagos 4, 4, 2

# Si todo está carísimo, nadie compra
assert grafo_preferidos(V, {i: 99 for i in VENDEDORES},
                        VENDEDORES, COMPRADORES).number_of_edges() == 0
ok("Ejercicio 1.1")

OK - Ejercicio 1.1


### Ejercicio 1.2 — Emparejamiento perfecto y conjunto constreñido (10 puntos)

Las dos mitades del teorema de Hall, como código.

**(a)** `emparejamiento_perfecto(G, compradores)`: diccionario `{comprador: vendedor}`
si existe, `None` si no. Usen `nx.bipartite.maximum_matching` con `top_nodes`.

**(b)** `conjunto_restringido(G, compradores)`: la tupla $(S, N(S))$ del primer
conjunto constreñido, o `None`. Para estos tamaños basta enumerar todos los
subconjuntos con `itertools.combinations`, de menor a mayor.

> **Cuidado con el caso degenerado:** un comprador sin aristas forma él solo un
> conjunto constreñido, con $|N(S)| = 0 < 1$.


In [28]:
def emparejamiento_perfecto(G, compradores):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


def conjunto_restringido(G, compradores):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError

In [29]:
def emparejamiento_perfecto(G, compradores):
    """Diccionario {comprador: vendedor} si existe emparejamiento perfecto, None si no."""
    matching = nx.bipartite.maximum_matching(G, top_nodes=compradores)
    if not all(j in matching for j in compradores):
        return None
    return {j: matching[j] for j in compradores}


def conjunto_restringido(G, compradores):
    """(S, N(S)) del primer conjunto constreñido (de menor a mayor tamaño), o None."""
    for r in range(1, len(compradores) + 1):
        for combo in itertools.combinations(compradores, r):
            S = set(combo)
            N = set()
            for j in S:
                N.update(G.neighbors(j))
            if len(N) < len(S):
                return S, N
    return None


In [30]:
# A precios cero no hay emparejamiento perfecto, y el certificado lo confirma
assert emparejamiento_perfecto(G0, COMPRADORES) is None
S, N = conjunto_restringido(G0, COMPRADORES)
assert len(N) < len(S) and N == {"a"}

# A precios (3,1,0) sí lo hay, luego no puede haber conjunto constreñido
M1 = emparejamiento_perfecto(G1, COMPRADORES)
assert M1 is not None and set(M1.values()) == set(VENDEDORES)
assert conjunto_restringido(G1, COMPRADORES) is None
ok("Ejercicio 1.2")

OK - Ejercicio 1.2


**Interpretación 1.2.** Las dos funciones nunca devuelven ambas un resultado no
trivial: es el teorema de Hall. Expliquen en 2–3 frases qué dirección de la
equivalencia usa cada uno de los dos `assert` de arriba.

> *TU RESPUESTA AQUÍ.*


### Ejercicio 1.3 — El procedimiento de subasta (10 puntos)

[EK] §10.4 demuestra la existencia de precios de equilibrio **exhibiendo un
algoritmo**, que tiene la forma de una subasta ascendente:

> 1. Empezar con todos los precios en 0.
> 2. Construir el grafo de vendedores preferidos.
> 3. Si hay emparejamiento perfecto, **terminar**.
> 4. Si no, encontrar un conjunto constreñido $S$ y subir en **una unidad** el precio
>    de cada vendedor de $N(S)$.
> 5. **Normalizar**: si el precio mínimo quedó por encima de 0, restarles a todos la
>    misma cantidad para que vuelva a ser 0.
> 6. Volver a 2.

La lógica del paso 4 es de exceso de demanda: los vendedores de $N(S)$ son los
sobre-demandados por el grupo $S$.

**El paso 5 merece atención.** Restar la misma constante a todos los precios **no
cambia el grafo de preferidos**, porque resta lo mismo a todos los pagos y no altera
cuál es el máximo. Pero **sí garantiza la terminación** — y vale la pena entender
exactamente por qué, porque no es obvio a primera vista.

Define el potencial $\Phi(p) = \sum_i p(i) + \sum_j u_j(p)$, donde $u_j(p)$ es el
mejor pago del comprador $j$ a precios $p$. Por dualidad débil del problema de
asignación, $\Phi(p) \geq \text{OPT}$ para **cualquier** $p$ (OPT = valoración total
óptima, una cota fija). Cada vez que el paso 4 sube en 1 el precio de cada vendedor de
$N(S)$, **todo** comprador de $S$ tenía sus vendedores óptimos dentro de $N(S)$ (así se
construyó $N(S)$), así que su mejor pago baja exactamente en 1. Luego

$$\Delta\Phi = \underbrace{|N(S)|}_{\text{sube } \sum p} - \underbrace{|S|}_{\text{baja } \sum u} < 0,$$

porque $S$ es constreñido. Con valoraciones enteras, $\Phi$ baja en al menos 1 en cada
ronda **en la que efectivamente se sube algún precio**, y como está acotado por abajo,
eso por sí solo ya acotaría el número de rondas — de hecho $\Phi$ es invariante a
sumarle una constante a todos los precios, así que este argumento **no depende** del
paso 5.

El problema real que resuelve el paso 5 es otro: sin normalizar, los precios **solo
suben** (el paso 4 nunca los baja), y pueden llegar a un punto en que el vendedor más
barato tenga un precio tan alto que **algún comprador tenga pago negativo con todas
las opciones**. Ese comprador queda sin ninguna arista — es exactamente el caso
degenerado del ejercicio 1.2 ($S = \{j\}$, $N(S) = \varnothing$, con
$|N(S)| = 0 < 1$). En ese momento el paso 4 no tiene nada que subir: $N(S)$ está
vacío, los precios no cambian, y el algoritmo queda **congelado para siempre** en ese
mismo vector de precios.

Normalizar evita justo eso: al restar la misma cantidad para que el precio mínimo
vuelva a ser 0, siempre queda al menos un vendedor gratis, y como las valoraciones son
$\geq 0$, ningún comprador puede quedar con pago negativo en todas las opciones —
nunca se aísla. Eso garantiza que $N(S) \neq \varnothing$ para cualquier $S$
constreñido, así que el paso 4 **siempre** logra subir al menos un precio, y ahí sí
aplica el argumento de $\Phi$ para acotar el número de rondas.

Implementen `precios_de_equilibrio(V, vendedores, compradores, max_iter=1000)`, que
devuelva `(p, M, rondas)`.


In [31]:
def precios_de_equilibrio(V, vendedores, compradores, max_iter=1000):
    """TODO"""
    # p = {i: 0 for i in vendedores}
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


#p_eq, M_eq, rondas = precios_de_equilibrio(V, VENDEDORES, COMPRADORES)
#print("Precios de equilibrio:", p_eq)
#print("Emparejamiento       :", M_eq)
#print("Rondas               :", rondas)

In [32]:
def precios_de_equilibrio(V, vendedores, compradores, max_iter=1000):
    """(p, M, rondas): precios de equilibrio, emparejamiento y número de rondas."""
    p = {i: 0 for i in vendedores}
    for ronda in range(max_iter):
        G = grafo_preferidos(V, p, vendedores, compradores)
        M = emparejamiento_perfecto(G, compradores)
        if M is not None:
            return p, M, ronda
        S, N = conjunto_restringido(G, compradores)
        for i in N:
            p[i] += 1
        minimo = min(p.values())
        if minimo > 0:
            for i in vendedores:
                p[i] -= minimo
    raise RuntimeError("precios_de_equilibrio no convergió en max_iter rondas")


p_eq, M_eq, rondas = precios_de_equilibrio(V, VENDEDORES, COMPRADORES)
print("Precios de equilibrio:", p_eq)
print("Emparejamiento       :", M_eq)
print("Rondas               :", rondas)


Precios de equilibrio: {'a': 3, 'b': 1, 'c': 0}
Emparejamiento       : {'x': 'a', 'y': 'c', 'z': 'b'}
Rondas               : 3


In [33]:
assert p_eq == {"a": 3, "b": 1, "c": 0}, "Debe reproducir el ejemplo del libro"
assert min(p_eq.values()) == 0, "Los precios deben quedar normalizados"
assert M_eq == {"x": "a", "y": "c", "z": "b"}

# Con valoraciones idénticas ningún precio tiene por qué subir
V2 = {("a", "x"): 5, ("b", "x"): 5, ("a", "y"): 5, ("b", "y"): 5}
p2, _, _ = precios_de_equilibrio(V2, ["a", "b"], ["x", "y"])
assert p2 == {"a": 0, "b": 0}
ok("Ejercicio 1.3")

OK - Ejercicio 1.3


### Ejercicio 1.4 — Optimalidad y no unicidad del equilibrio (8 puntos)

El teorema dice que los precios de equilibrio **existen**, no que sean únicos. De
hecho hay muchos: el libro exhibe dos vectores distintos para estas mismas
valoraciones (figuras 10.5(b) y 10.5(d)).

Se les dan ya escritas `valoracion_total` y `asignacion_optima` (esta última por fuerza
bruta sobre las $n!$ permutaciones). Lean el código de la celda siguiente antes de
continuar: van a usarlo.

Implementen únicamente `todos_los_equilibrios(V, vendedores, compradores, pmax=12)`,
que enumere todos los vectores de precios **enteros y normalizados** (mínimo igual a 0)
con $0 \leq p(i) \leq p_{max}$ que sean de equilibrio, y los devuelva como lista
ordenada de tuplas.

Luego reporten, para el mínimo y el máximo del conjunto: la valoración total, el pago
total de los compradores y el ingreso total de los vendedores.


In [34]:
# Funciones dadas: no hay que modificarlas, pero sí entenderlas.

def valoracion_total(V, M):
    """Suma de v(M(j), j) sobre los compradores."""
    return sum(V[(i, j)] for j, i in M.items())


def asignacion_optima(V, vendedores, compradores):
    """(valor, asignación) óptima por fuerza bruta sobre las n! permutaciones."""
    mejor_valor, mejor_M = None, None
    for perm in itertools.permutations(vendedores):
        M = dict(zip(compradores, perm))
        valor = valoracion_total(V, M)
        if mejor_valor is None or valor > mejor_valor:
            mejor_valor, mejor_M = valor, M
    return mejor_valor, mejor_M


v_opt, M_opt = asignacion_optima(V, VENDEDORES, COMPRADORES)
print(f"Asignación óptima: {M_opt} -> valoración {v_opt}")

Asignación óptima: {'x': 'a', 'y': 'c', 'z': 'b'} -> valoración 23


In [35]:
def todos_los_equilibrios(V, vendedores, compradores, pmax=12):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


equilibrios = todos_los_equilibrios(V, VENDEDORES, COMPRADORES)

print(f"Vectores de precios de equilibrio: {len(equilibrios)}")
print(f"  mínimo: {equilibrios[0]}   máximo: {equilibrios[-1]}\n")

for combo in [equilibrios[0], equilibrios[-1]]:
    p = dict(zip(VENDEDORES, combo))
    M = emparejamiento_perfecto(grafo_preferidos(V, p, VENDEDORES, COMPRADORES),
                                COMPRADORES)
    valor = valoracion_total(V, M)
    pago_compradores = sum(V[(M[j], j)] - p[M[j]] for j in COMPRADORES)
    print(f"  precios {combo}: valoración {valor} | "
          f"compradores {pago_compradores} | vendedores {sum(combo)}")

NotImplementedError: 

In [38]:
def todos_los_equilibrios(V, vendedores, compradores, pmax=12):
    """Lista ordenada de tuplas de precios enteros, normalizados (mínimo 0), de equilibrio."""
    resultados = []
    for combo in itertools.product(range(pmax + 1), repeat=len(vendedores)):
        if min(combo) != 0:
            continue
        p = dict(zip(vendedores, combo))
        G = grafo_preferidos(V, p, vendedores, compradores)
        if emparejamiento_perfecto(G, compradores) is not None:
            resultados.append(combo)
    return sorted(resultados)

equilibrios = todos_los_equilibrios(V, VENDEDORES, COMPRADORES)

print(f"Vectores de precios de equilibrio: {len(equilibrios)}")
print(f"  mínimo: {equilibrios[0]}   máximo: {equilibrios[-1]}\n")

for combo in [equilibrios[0], equilibrios[-1]]:
    p = dict(zip(VENDEDORES, combo))
    M = emparejamiento_perfecto(grafo_preferidos(V, p, VENDEDORES, COMPRADORES),
                                COMPRADORES)
    valor = valoracion_total(V, M)
    pago_compradores = sum(V[(M[j], j)] - p[M[j]] for j in COMPRADORES)
    print(f"  precios {combo}: valoración {valor} | "
          f"compradores {pago_compradores} | vendedores {sum(combo)}")


Vectores de precios de equilibrio: 20
  mínimo: (3, 1, 0)   máximo: (10, 3, 0)

  precios (3, 1, 0): valoración 23 | compradores 19 | vendedores 4
  precios (10, 3, 0): valoración 23 | compradores 10 | vendedores 13


In [39]:
assert v_opt == 23
assert valoracion_total(V, M_eq) == v_opt, "El equilibrio debe dar el óptimo social"

# No unicidad: hay veinte vectores de equilibrio normalizados
assert len(equilibrios) == 20
assert equilibrios[0] == (3, 1, 0), "El mínimo es el que produce la subasta"
assert equilibrios[-1] == (10, 3, 0)

# Todos alcanzan el óptimo social, pero reparten distinto
for combo in equilibrios:
    p = dict(zip(VENDEDORES, combo))
    M = emparejamiento_perfecto(grafo_preferidos(V, p, VENDEDORES, COMPRADORES),
                                COMPRADORES)
    assert valoracion_total(V, M) == v_opt

# El conjunto es una retícula: ínfimo y supremo son ellos mismos de equilibrio
infimo = tuple(min(c[k] for c in equilibrios) for k in range(3))
supremo = tuple(max(c[k] for c in equilibrios) for k in range(3))
assert infimo in equilibrios and supremo in equilibrios
ok("Ejercicio 1.4")

OK - Ejercicio 1.4


**Interpretación 1.4 (obligatoria).** Respondan en 4–5 frases:

1. Los veinte vectores dan la misma valoración total (23) pero repartos muy distintos
   entre compradores y vendedores. ¿Qué determina la teoría y qué deja abierto?
2. El procedimiento de subasta del ejercicio 1.3 termina siempre en el vector
   **mínimo**. ¿Por qué? ¿A quién favorece esa selección?
3. [EK] §10.5 observa que la subasta de un solo bien es el caso particular con un
   vendedor real y $n-1$ vendedores ficticios de valoración cero. Si las valoraciones
   son 10, 7, 5 y 3, ¿qué precios son de equilibrio y en cuál se detiene el
   procedimiento? Relacionen con la subasta de segundo precio.

> *TU RESPUESTA AQUÍ.*


---
# Parte 2 — Redes de comercio con intermediarios ([EK] cap. 11) · 28 puntos

## 2.0 Teoría

### Qué le falta al capítulo 10

El capítulo 10 muestra que los precios de equilibrio existen y son eficientes, pero
**no dice quién los pone**. En muchos mercados reales compradores y vendedores no
interactúan directamente sino a través de **intermediarios** (*traders*), y no todos
tienen acceso a los mismos.

### El modelo ([EK] §11.2)

- Cada **vendedor** $i$ tiene una unidad que valora en $v_i$ (en los ejemplos, 0).
- Cada **comprador** $j$ valora una unidad en $v_j$; unidades adicionales le valen 0.
- Los **intermediarios** no valoran el bien: compran para revender.

Restricción estructural clave: **toda arista une a un intermediario con un comprador o
con un vendedor**. Nadie comercia directamente con su contraparte final.

Juego de dos etapas:

1. Cada intermediario $t$ anuncia un precio de compra $b_{ti}$ (*bid*) a cada vendedor
   vecino y un precio de venta $a_{tj}$ (*ask*) a cada comprador vecino.
2. Vistos los precios, cada comprador y vendedor elige a lo sumo un intermediario.

| Agente | Pago |
|---|---|
| Vendedor $i$ que vende a $t$ | $b_{ti}$ |
| Comprador $j$ que compra a $t$ | $v_j - a_{tj}$ |
| Intermediario $t$ | $\sum(\text{asks aceptados}) - \sum(\text{bids aceptados})$ |

El concepto de solución es el **equilibrio de Nash perfecto en subjuegos**.

### Los casos extremos

- **Monopolio** (fig. 11.6): un vendedor, un comprador, un intermediario. Único
  equilibrio: $b = 0$, $a = 1$. Se queda con todo.
- **Competencia perfecta** (fig. 11.7): el mismo par con **dos** intermediarios. En
  equilibrio ambos ponen $b = a = x$ y su beneficio es **cero**.
- **Competencia perfecta implícita** (fig. 11.9): ningún intermediario compite
  directamente con otro por el mismo par, y aun así todos ganan cero.

### Bienestar y el criterio de arista esencial

Cada unidad que se mueve de $i$ a $j$ aporta $v_j - v_i$. Los precios se cancelan al
sumar los tres pagos de una transacción:

$$(b_{ti} - v_i) + (a_{tj} - b_{ti}) + (v_j - a_{tj}) = v_j - v_i,$$

de modo que **el dinero es una pura transferencia**. Todo equilibrio produce un flujo
socialmente óptimo ([EK] §11.5). La pregunta fina es quién captura el excedente:

> **Arista esencial.** Una arista cuya eliminación **cambia el valor del óptimo
> social**.
>
> **Teorema ([EK] §11.6).** Un intermediario obtiene beneficio positivo en **algún**
> equilibrio si y solo si tiene al menos una arista esencial.

El criterio es sobre **aristas**, no sobre nodos: en la fig. 11.13, eliminar el nodo
$T_1$ sí reduciría el óptimo social, pero ninguna arista individual lo hace, y por eso
$T_1$ gana cero pese a monopolizar a sus vendedores.


In [40]:
# Redes del capítulo 11. Formato: aristas (agente, intermediario).
RED_MONOPOLIO = {                                            # Figura 11.6
    "aristas": [("S1", "T1"), ("B1", "T1")],
    "vendedores": ["S1"], "valores": {"B1": 1}}

RED_COMPETENCIA = {                                          # Figura 11.7
    "aristas": [("S1", "T1"), ("S1", "T2"), ("B1", "T1"), ("B1", "T2")],
    "vendedores": ["S1"], "valores": {"B1": 1}}

RED_11_8 = {                                                 # Figura 11.8
    "aristas": [("S1", "T1"), ("S2", "T1"), ("S2", "T2"), ("S3", "T2"),
                ("B1", "T1"), ("B2", "T1"), ("B2", "T2"), ("B3", "T2")],
    "vendedores": ["S1", "S2", "S3"], "valores": {"B1": 1, "B2": 1, "B3": 1}}

RED_11_9 = {                                                 # Figura 11.9
    "aristas": [("S1", "T1"), ("S1", "T2"), ("S2", "T3"), ("S2", "T4"),
                ("B1", "T1"), ("B1", "T3"), ("B2", "T2"), ("B2", "T4")],
    "vendedores": ["S1", "S2"], "valores": {"B1": 1, "B2": 1}}

RED_11_13 = {                                                # Figura 11.13
    "aristas": [("S1", "T1"), ("S2", "T1"), ("S3", "T2"),
                ("B1", "T1"), ("B2", "T1"), ("B1", "T2"), ("B2", "T2")],
    "vendedores": ["S1", "S2", "S3"], "valores": {"B1": 1, "B2": 1}}

REDES = [("Monopolio (11.6)", RED_MONOPOLIO), ("Competencia (11.7)", RED_COMPETENCIA),
         ("Figura 11.8", RED_11_8), ("Figura 11.9", RED_11_9),
         ("Figura 11.13", RED_11_13)]

for nombre, red in REDES:
    traders = {t for _, t in red["aristas"]}
    print(f"{nombre:22s} {len(red['vendedores'])} vendedores, "
          f"{len(traders)} intermediarios, {len(red['valores'])} compradores")

Monopolio (11.6)       1 vendedores, 1 intermediarios, 1 compradores
Competencia (11.7)     1 vendedores, 2 intermediarios, 1 compradores
Figura 11.8            3 vendedores, 2 intermediarios, 3 compradores
Figura 11.9            2 vendedores, 4 intermediarios, 2 compradores
Figura 11.13           3 vendedores, 2 intermediarios, 2 compradores


### Ejercicio 2.1 — Pagos en la red de comercio (8 puntos)

Implementen `pagos(red, precios, flujo)` según la tabla de arriba, donde `precios` es
`{(intermediario, agente): precio}` y `flujo` es una lista de tuplas
`(vendedor, intermediario, comprador)`, una por unidad movida.

Verifiquen contra la figura 11.4(b), donde el texto reporta beneficios de 0.6 para
$T_1$ y 1.4 para $T_2$.


In [ ]:
def pagos(red, precios, flujo):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


precios_11_4 = {("T1", "S1"): 0.2, ("T1", "B1"): 0.8,
                ("T1", "S2"): 0.2, ("T1", "B2"): 0.8,
                ("T2", "S2"): 0.3, ("T2", "B2"): 0.7,
                ("T2", "S3"): 0.0, ("T2", "B3"): 1.0}
flujo_11_4 = [("S1", "T1", "B1"), ("S2", "T2", "B2"), ("S3", "T2", "B3")]

resultado = pagos(RED_11_8, precios_11_4, flujo_11_4)
for agente in ["S1", "S2", "S3", "T1", "T2", "B1", "B2", "B3"]:
    print(f"  {agente}: {round(resultado.get(agente, 0), 4)}")

In [41]:
def pagos(red, precios, flujo):
    """{agente: pago} para vendedores, intermediarios y compradores según el flujo."""
    resultado = {}
    for vendedor, t, comprador in flujo:
        b = precios[(t, vendedor)]      # bid: lo que t le paga al vendedor
        a = precios[(t, comprador)]     # ask: lo que t le cobra al comprador
        resultado[vendedor] = resultado.get(vendedor, 0) + b
        resultado[comprador] = resultado.get(comprador, 0) + (red["valores"][comprador] - a)
        resultado[t] = resultado.get(t, 0) + (a - b)
    return resultado

precios_11_4 = {("T1", "S1"): 0.2, ("T1", "B1"): 0.8,
                ("T1", "S2"): 0.2, ("T1", "B2"): 0.8,
                ("T2", "S2"): 0.3, ("T2", "B2"): 0.7,
                ("T2", "S3"): 0.0, ("T2", "B3"): 1.0}
flujo_11_4 = [("S1", "T1", "B1"), ("S2", "T2", "B2"), ("S3", "T2", "B3")]

resultado = pagos(RED_11_8, precios_11_4, flujo_11_4)
for agente in ["S1", "S2", "S3", "T1", "T2", "B1", "B2", "B3"]:
    print(f"  {agente}: {round(resultado.get(agente, 0), 4)}")


  S1: 0.2
  S2: 0.3
  S3: 0.0
  T1: 0.6
  T2: 1.4
  B1: 0.2
  B2: 0.3
  B3: 0.0


In [42]:
assert abs(resultado["T1"] - 0.6) < 1e-9, "T1 debe ganar 0.6 (texto de EK 11.2)"
assert abs(resultado["T2"] - 1.4) < 1e-9, "T2 debe ganar 1.4"
assert abs(resultado["B1"] - 0.2) < 1e-9 and abs(resultado["S2"] - 0.3) < 1e-9
# El excedente se reparte por completo: 3 unidades con v_j - v_i = 1 cada una
assert abs(sum(resultado.values()) - 3.0) < 1e-9
ok("Ejercicio 2.1")

OK - Ejercicio 2.1


### Ejercicio 2.2 — El óptimo social como problema de flujo (10 puntos)

Como los vendedores valoran el bien en 0, el bienestar social es $\sum v_j$ sobre los
compradores servidos. Maximizarlo sujeto a la red es un **flujo de costo mínimo**:

```
FUENTE → vendedores → intermediarios → compradores → SUMIDERO
```

con capacidad 1 en cada arista de la red y **sin límite** en los intermediarios, que es
exactamente el supuesto del modelo.

Implementen `optimo_social(red)` con `nx.max_flow_min_cost`. Dos detalles:

- Para **maximizar** valor con un algoritmo que **minimiza** costo, pongan el valor del
  comprador con **signo negativo** en la arista `comprador → SUMIDERO`.
- `nx.max_flow_min_cost` requiere pesos **enteros**: multipliquen por una escala y
  dividan al final.


In [ ]:
ESCALA = 1000

def optimo_social(red):
    """TODO"""
    D = nx.DiGraph()
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


for nombre, red in REDES:
    print(f"{nombre:22s} óptimo social = {optimo_social(red)}")

In [43]:
ESCALA = 1000

def optimo_social(red):
    """Bienestar social óptimo: suma de valoraciones de los compradores servidos."""
    D = nx.DiGraph()
    for v in red["vendedores"]:
        D.add_edge("FUENTE", v, capacity=1, weight=0)
    for agente, t in red["aristas"]:
        if agente in red["vendedores"]:
            D.add_edge(agente, t, capacity=1, weight=0)
        else:
            D.add_edge(t, agente, capacity=1, weight=0)
    for j, valor in red["valores"].items():
        D.add_edge(j, "SUMIDERO", capacity=1, weight=-round(valor * ESCALA))

    flujo = nx.max_flow_min_cost(D, "FUENTE", "SUMIDERO")
    costo = nx.cost_of_flow(D, flujo)
    return -costo / ESCALA


for nombre, red in REDES:
    print(f"{nombre:22s} óptimo social = {optimo_social(red)}")


Monopolio (11.6)       óptimo social = 1.0
Competencia (11.7)     óptimo social = 1.0
Figura 11.8            óptimo social = 3.0
Figura 11.9            óptimo social = 2.0
Figura 11.13           óptimo social = 2.0


In [44]:
assert optimo_social(RED_MONOPOLIO) == 1.0
assert optimo_social(RED_COMPETENCIA) == 1.0
assert optimo_social(RED_11_8) == 3.0
assert optimo_social(RED_11_9) == 2.0
assert optimo_social(RED_11_13) == 2.0

# Figura 11.11: añadir UNA arista rompe el cuello de botella y sube el óptimo de 7 a 9
RED_11_11a = {"aristas": [("S1", "T1"), ("S2", "T1"), ("S3", "T2"),
                          ("B1", "T1"), ("B2", "T1"), ("B2", "T2"),
                          ("B3", "T2"), ("B4", "T2")],
              "vendedores": ["S1", "S2", "S3"],
              "valores": {"B1": 1, "B2": 2, "B3": 3, "B4": 4}}
RED_11_11b = {**RED_11_11a, "aristas": RED_11_11a["aristas"] + [("S2", "T2")]}
assert optimo_social(RED_11_11a) == 7.0
assert optimo_social(RED_11_11b) == 9.0
ok("Ejercicio 2.2")

OK - Ejercicio 2.2


### Ejercicio 2.3 — Aristas esenciales y beneficio (10 puntos)

Implementen el criterio del teorema de [EK] §11.6:

- `aristas_esenciales(red)`: aristas cuya eliminación **reduce** `optimo_social`.
- `intermediarios_con_beneficio(red)`: intermediarios con al menos una arista esencial.


In [ ]:
def aristas_esenciales(red):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


def intermediarios_con_beneficio(red):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


for nombre, red in REDES:
    print(f"{nombre:22s} esenciales: {aristas_esenciales(red)}")
    print(f"{'':22s} con beneficio: {intermediarios_con_beneficio(red) or '{}'}")

In [45]:
def aristas_esenciales(red):
    """Aristas cuya eliminación reduce el óptimo social."""
    optimo = optimo_social(red)
    esenciales = []
    for arista in red["aristas"]:
        red_sin = {**red, "aristas": [a for a in red["aristas"] if a != arista]}
        if optimo_social(red_sin) < optimo:
            esenciales.append(arista)
    return esenciales


def intermediarios_con_beneficio(red):
    """Intermediarios con al menos una arista esencial."""
    return {t for _, t in aristas_esenciales(red)}

for nombre, red in REDES:
    print(f"{nombre:22s} esenciales: {aristas_esenciales(red)}")
    print(f"{'':22s} con beneficio: {intermediarios_con_beneficio(red) or '{}'}")


Monopolio (11.6)       esenciales: [('S1', 'T1'), ('B1', 'T1')]
                       con beneficio: {'T1'}
Competencia (11.7)     esenciales: []
                       con beneficio: {}
Figura 11.8            esenciales: [('S1', 'T1'), ('S3', 'T2'), ('B1', 'T1'), ('B3', 'T2')]
                       con beneficio: {'T2', 'T1'}
Figura 11.9            esenciales: []
                       con beneficio: {}
Figura 11.13           esenciales: []
                       con beneficio: {}


In [46]:
assert intermediarios_con_beneficio(RED_MONOPOLIO) == {"T1"}
assert intermediarios_con_beneficio(RED_COMPETENCIA) == set()
assert intermediarios_con_beneficio(RED_11_8) == {"T1", "T2"}
assert set(aristas_esenciales(RED_11_8)) == {("S1", "T1"), ("S3", "T2"),
                                             ("B1", "T1"), ("B3", "T2")}
assert intermediarios_con_beneficio(RED_11_9) == set(), "Competencia perfecta implícita"
assert intermediarios_con_beneficio(RED_11_13) == set(), "Monopolio sin beneficio"
ok("Ejercicio 2.3")

OK - Ejercicio 2.3


**Interpretación 2.3 (obligatoria).** En la figura **11.13** cada intermediario
monopoliza a sus vendedores y gana cero. En la figura **11.9** ninguno compite
directamente con otro y también ganan cero.

Respondan en 4–5 frases:
1. ¿Qué tienen en común esas dos redes según el criterio de arista esencial?
2. ¿Por qué el **grado** de un intermediario no predice su beneficio? Citen un número
   de la salida anterior.
3. En la 11.13, ¿qué le pasaría al óptimo social si eliminaran el **nodo** $T_2$
   completo? ¿Por qué el teorema se formula sobre aristas y no sobre nodos?

> *TU RESPUESTA AQUÍ.*


---
# Parte 3 — Negociación y poder ([EK] cap. 12) · 26 puntos

## 3.0 Teoría

### El diseño experimental

Se toma un grafo pequeño, una persona por nodo, y se coloca **\$1 sobre cada arista**
para repartir entre sus extremos. Rige la **regla de un intercambio**: cada nodo puede
cerrar trato con **a lo sumo un** vecino. Los intercambios realizados forman un
emparejamiento, no necesariamente perfecto: algunos nodos quedan **excluidos** y se van
sin nada. La exclusión es el mecanismo del poder.

Resultados experimentales ([EK] §12.3):

| Red | Qué ocurre |
|---|---|
| $A - B$ | Reparto aproximadamente igual |
| $A - B - C$ | **Poder fuerte** de $B$: puede excluir a cualquiera y ninguno tiene alternativa |
| $A-B-C-D$ | **Poder débil** de $B$: para excluir a $A$ debe tratar con $C$, que sí tiene alternativa |
| $A-B-C-D-E$ | El nodo "central" $C$ resulta **débil**: sus dos socios tienen alternativas atractivas |

El 5-camino es el ejemplo a retener: **la centralidad puede ser una medida engañosa del
poder**.

### Los tres conceptos

> **Solución de negociación de Nash ([EK] §12.5).** Si $A$ tiene opción externa $x$ y
> $B$ tiene $y$, con $x + y \leq 1$, el excedente $s = 1 - x - y$ se reparte por
> mitades: $A$ recibe $x + s/2$ y $B$ recibe $y + s/2$.

> **Resultado.** Un emparejamiento más un valor por nodo: los emparejados suman 1, los
> excluidos valen 0.
>
> **Inestabilidad ([EK] §12.7).** Una arista **fuera** del emparejamiento cuyos
> extremos suman **menos de 1**: tienen la oportunidad y el incentivo de romper sus
> acuerdos. Un resultado es **estable** si no tiene inestabilidades.

> **Opción externa.** Lo máximo que un nodo podría obtener robándole un vecino a su
> pareja: el máximo de $1 - (\text{valor del vecino})$ sobre los vecinos que no son su
> pareja, y $0$ si no hay ninguno.
>
> **Resultado balanceado ([EK] §12.8).** Un resultado estable en el que, para **cada**
> arista del emparejamiento, el reparto es exactamente la solución de Nash con esas
> opciones externas.

La estabilidad explica el poder fuerte (en el 3-camino, $B$ se lleva todo) y la
ausencia de equilibrio (el **triángulo** no admite ningún resultado estable). Pero es
demasiado permisiva en el poder débil: el 4-camino admite decenas de resultados
estables, incluido $(\tfrac12,\tfrac12,\tfrac12,\tfrac12)$, que los experimentos no
producen. El balance lo refina.


### Ejercicio 3.1 — Solución de Nash y opción externa (6 puntos)

Implementen `nash_bargaining(x, y)` —que lance `ValueError` si $x + y > 1$— y
`opcion_externa(G, M, valores, nodo)` según las definiciones de arriba.


In [ ]:
def nash_bargaining(x, y):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


def opcion_externa(G, M, valores, nodo):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError

In [ ]:
assert nash_bargaining(Fraction(0), Fraction(0)) == (Fraction(1, 2), Fraction(1, 2))
assert nash_bargaining(Fraction(0), Fraction(1, 2)) == (Fraction(1, 4), Fraction(3, 4))
try:
    nash_bargaining(Fraction(3, 4), Fraction(1, 2))
    raise AssertionError("Debió lanzar ValueError")
except ValueError:
    pass

# Figura 12.8(a): el reparto de todos-1/2 en el 4-camino
P4 = nx.path_graph(["A", "B", "C", "D"])
M_ext = [("A", "B"), ("C", "D")]
val_med = {n: Fraction(1, 2) for n in P4}
assert opcion_externa(P4, M_ext, val_med, "B") == Fraction(1, 2)   # puede robarle C a D
assert opcion_externa(P4, M_ext, val_med, "A") == 0                # no tiene alternativa
ok("Ejercicio 3.1")

### Ejercicio 3.2 — Resultados estables (10 puntos)

Implementen:

- `es_estable(G, M, valores)`.
- `emparejamientos_maximales(G)`: los emparejamientos a los que no se les puede añadir
  otra arista. Con 3–5 nodos basta enumerar subconjuntos de aristas y filtrar.
- `buscar_estables(G, denom=12)`: para cada emparejamiento maximal, prueba repartos
  sobre la grilla $0, \tfrac1d, \dots, 1$ en cada arista emparejada.


In [ ]:
def es_estable(G, M, valores):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


def emparejamientos_maximales(G):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


def buscar_estables(G, denom=12):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


REDES_12 = {
    "2-camino":  nx.path_graph(["A", "B"]),
    "3-camino":  nx.path_graph(["A", "B", "C"]),
    "4-camino":  nx.path_graph(["A", "B", "C", "D"]),
    "5-camino":  nx.path_graph(["A", "B", "C", "D", "E"]),
    "triángulo": nx.cycle_graph(["A", "B", "C"]),
    "stem":      nx.Graph([("A", "B"), ("B", "C"), ("B", "D"), ("C", "D")]),
}

for nombre, g in REDES_12.items():
    print(f"{nombre:10s} resultados estables: {len(buscar_estables(g))}")

In [ ]:
# El triángulo no admite ningún resultado estable ([EK] §12.7, figura 12.4)
assert buscar_estables(REDES_12["triángulo"]) == []

# 3-camino: B se queda con todo. Poder fuerte
est3 = buscar_estables(REDES_12["3-camino"])
assert len(est3) > 0 and all(v["B"] == 1 for _, v in est3)

# 5-camino: B y D se llevan todo; el nodo "central" C obtiene 0
est5 = buscar_estables(REDES_12["5-camino"])
assert all(v["B"] == 1 and v["D"] == 1 and v["C"] == 0 for _, v in est5)

# 4-camino: la estabilidad sola es demasiado permisiva
assert len(buscar_estables(REDES_12["4-camino"])) > 50
ok("Ejercicio 3.2")

**Interpretación 3.2.** El nodo $C$ del 5-camino es el más central según
intermediación y obtiene 0 en **todo** resultado estable. Expliquen por qué en términos
de **las alternativas de sus vecinos**, no de las suyas propias (2–3 frases).

> *TU RESPUESTA AQUÍ.*


### Ejercicio 3.3 — Resultados balanceados (10 puntos)

Implementen `es_balanceado(G, M, valores)` y `buscar_balanceados(G, denom=12)`.
Deben reproducir los valores canónicos del libro: $(\tfrac13,\tfrac23,\tfrac23,
\tfrac13)$ en el 4-camino (fig. 12.8(b)) y $A=\tfrac14$, $B=\tfrac34$,
$C=D=\tfrac12$ en el grafo *stem* (fig. 12.9).


In [ ]:
def es_balanceado(G, M, valores):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


def buscar_balanceados(G, denom=12):
    """TODO"""
    # TU CÓDIGO AQUÍ
    raise NotImplementedError


for nombre, g in REDES_12.items():
    bal = buscar_balanceados(g)
    if not bal:
        print(f"{nombre:10s} -> ninguno")
        continue
    _, v = bal[0]
    print(f"{nombre:10s} -> {len(bal)}; ejemplo: {({n: str(v[n]) for n in sorted(v)})}")

In [ ]:
def reparto(G):
    bal = buscar_balanceados(G)
    assert bal, "Se esperaba al menos un resultado balanceado"
    return bal[0][1]

assert reparto(REDES_12["2-camino"])["A"] == Fraction(1, 2)

# 4-camino: figura 12.8(b), y es único
r4 = reparto(REDES_12["4-camino"])
assert (r4["A"], r4["B"], r4["C"], r4["D"]) == (Fraction(1, 3), Fraction(2, 3),
                                                Fraction(2, 3), Fraction(1, 3))
assert len(buscar_balanceados(REDES_12["4-camino"])) == 1

# Grafo stem: figura 12.9
r_stem = reparto(REDES_12["stem"])
assert (r_stem["A"], r_stem["B"]) == (Fraction(1, 4), Fraction(3, 4))
assert r_stem["C"] == r_stem["D"] == Fraction(1, 2)

# Sin estables tampoco hay balanceados
assert buscar_balanceados(REDES_12["triángulo"]) == []
ok("Ejercicio 3.3")

**Interpretación 3.3.** En el 4-camino $B$ obtiene $2/3$; en el *stem* obtiene
$3/4$. En ambas redes $B$ tiene poder **débil** sobre $A$. Expliquen la diferencia
calculando explícitamente la **opción externa de $B$** en cada caso y diciendo de dónde
sale (2–3 frases).

> *TU RESPUESTA AQUÍ.*


---
# Parte 4 — Discusión y cierre · 10 puntos

**4.1 (6 puntos).** En **máximo 350 palabras**:

> Los tres capítulos coinciden en un patrón: **la teoría determina la eficiencia y deja
> abierto el reparto**. En el cap. 10 hay veinte vectores de precios de equilibrio, todos
> con valoración total 23. En el cap. 11, bajo competencia perfecta, el equilibrio fija
> $b = a = x$ con $x$ indeterminado en $[0,1]$. En el cap. 12, el 4-camino admite decenas
> de resultados estables.
>
> (a) ¿Qué tienen en común esas tres indeterminaciones? ¿Es el mismo fenómeno?
> (b) El cap. 12 responde con un **refinamiento** (balance) que selecciona un resultado
> único. ¿Podría aplicarse una idea análoga al cap. 10 para seleccionar un vector de
> precios? Argumenten a favor o en contra, teniendo en cuenta cuál selecciona de hecho
> el procedimiento de subasta.
> (c) Den un ejemplo, tomado de sus propios resultados, de un nodo que **parece**
> poderoso por su grado o su centralidad y **no lo es**. Citen el número que lo
> demuestra.

> *TU RESPUESTA AQUÍ.*

---

**4.2 (4 puntos).** Declaración de Uso de IA completa y celda de cierre ejecutada.


In [ ]:
# Celda de cierre: no la modifiquen.
assert len(GRUPO["integrantes"]) >= 2, "El grupo debe tener mínimo 2 integrantes"
assert "Nombre Apellido" not in " ".join(GRUPO["integrantes"]), "Actualicen los nombres"
assert len(DECLARACION_IA.strip()) > 120, "La Declaración de Uso de IA está incompleta"
print("Taller 2 completado por:", ", ".join(GRUPO["integrantes"]))
print("Recuerden: Kernel -> Restart & Run All antes de comprimir el .zip")

---
### Antes de entregar — lista de chequeo

- [ ] `Kernel → Restart & Run All` corre sin errores.
- [ ] Las 10 celdas de verificación imprimen `OK`.
- [ ] Las 6 preguntas de interpretación están respondidas.
- [ ] `GRUPO` y `DECLARACION_IA` completos.
- [ ] `requirements.txt` y `README.md` incluidos en el `.zip`.

### Referencia

Easley, D. y Kleinberg, J. (2010). *Networks, Crowds, and Markets*, capítulos 10, 11 y
12. Cambridge University Press.
https://www.cs.cornell.edu/home/kleinber/networks-book/
